# Getting started with PyBaMM

PyBaMM (Python Battery Mathematical Modelling) is three things:

1. a **framework** for writing and solving systems of differential equations,
2. a **library** of battery models and parameter sets, and
3. tools for simulating **experiments** and visualising the results.

This notebook is a whistle-stop tour of all three. It condenses the
[getting started tutorials](https://docs.pybamm.org/en/latest/source/examples/index.html)
and the opening of the Oxford RSE
[PyBaMM model development course](https://train.rse.ox.ac.uk/material/HPCu/libraries/pybamm-developer).

In [1]:
import matplotlib.pyplot as plt
import numpy as np

import pybamm

pybamm.__version__

'26.9.0.0'

## 1. Run a model

Every simulation starts from a model. Here we use the Doyle-Fuller-Newman (DFN)
model with its default parameters, and discharge it for one hour.

In [2]:
model = pybamm.lithium_ion.DFN()
sim = pybamm.Simulation(model)
sim.solve([0, 3600])
sim.plot()

interactive(children=(FloatSlider(value=0.0, description='t', max=1.0, step=0.01), Output()), _dom_classes=('w…

## 2. Compare models

PyBaMM ships a hierarchy of models. The Single Particle Model (SPM) and the SPM
with electrolyte (SPMe) are reduced versions of the DFN that are much cheaper to
solve.

In [3]:
models = [
    pybamm.lithium_ion.SPM(),
    pybamm.lithium_ion.SPMe(),
    pybamm.lithium_ion.DFN(),
]
solutions = [pybamm.Simulation(model).solve([0, 3600]) for model in models]
pybamm.dynamic_plot(solutions)

interactive(children=(FloatSlider(value=0.0, description='t', max=1.0, step=0.01), Output()), _dom_classes=('w…

## 3. Read the outputs

A solution behaves like a dictionary of variables. Calling a variable with a
time (or an array of times, in seconds) interpolates it there. Prefer this to
the raw `.data` arrays, which are tied to the solver's internal time steps.

In [4]:
solution = solutions[-1]
t = np.linspace(0, 3600, 7)
solution["Voltage [V]"](t)

array([3.7717806 , 3.69346578, 3.65263611, 3.61325914, 3.59325807,
       3.57072289, 3.17168736])

Not sure what a variable is called? Search for it.

In [5]:
model.variables.search("electrolyte concentration")

Results for 'electrolyte concentration': ['Electrolyte concentration concatenation [mol.m-3]', 'Negative electrolyte concentration [mol.m-3]', 'X-averaged negative electrolyte concentration [mol.m-3]', 'Separator electrolyte concentration [mol.m-3]', 'X-averaged separator electrolyte concentration [mol.m-3]', 'Positive electrolyte concentration [mol.m-3]', 'X-averaged positive electrolyte concentration [mol.m-3]', 'Negative electrolyte concentration [Molar]', 'X-averaged negative electrolyte concentration [Molar]', 'Separator electrolyte concentration [Molar]', 'X-averaged separator electrolyte concentration [Molar]', 'Positive electrolyte concentration [Molar]', 'X-averaged positive electrolyte concentration [Molar]', 'Electrolyte concentration [mol.m-3]', 'X-averaged electrolyte concentration [mol.m-3]', 'Electrolyte concentration [Molar]', 'X-averaged electrolyte concentration [Molar]']


## 4. Change the parameters

Parameter sets live in `pybamm.ParameterValues`. You can load a published set,
search it, and change any value.

In [6]:
parameter_values = pybamm.ParameterValues("Chen2020")
parameter_values.search("electrode thickness")

Results for 'electrode thickness': ['Negative electrode thickness [m]', 'Positive electrode thickness [m]']
Negative electrode thickness [m] -> 8.52e-05
Positive electrode thickness [m] -> 7.56e-05


In [7]:
fig, ax = plt.subplots()
for current in [2.5, 5, 10]:
    parameter_values["Current function [A]"] = current
    sim = pybamm.Simulation(model, parameter_values=parameter_values)
    sol = sim.solve([0, 3600])
    ax.plot(sol.t / 60, sol["Voltage [V]"](sol.t), label=f"{current} A")
ax.set_xlabel("Time [min]")
ax.set_ylabel("Voltage [V]")
ax.legend()

## 5. Run an experiment

Real tests are sequences of steps. `pybamm.Experiment` turns plain-English steps
into a simulation protocol.

In [8]:
experiment = pybamm.Experiment(
    [
        (
            "Discharge at C/2 until 3.0V",
            "Rest for 1 hour",
            "Charge at 1C until 4.1V",
            "Hold at 4.1V until 50 mA",
        )
    ]
    * 2
)
sim = pybamm.Simulation(
    model, parameter_values=pybamm.ParameterValues("Chen2020"), experiment=experiment
)
sol = sim.solve()
sol.plot(["Current [A]", "Voltage [V]"])

interactive(children=(FloatSlider(value=0.0, description='t', max=12.68068514700335, step=0.1268068514700335),…

Each cycle and step of the experiment can be pulled out on its own.

In [9]:
rest = sol.cycles[0].steps[1]
rest["Voltage [V]"](rest.t[[0, -1]])

array([3.09435998, 3.17402627])

## 6. Under the hood: models are equations

PyBaMM models are written as symbolic expression trees, which are then
discretised and handed to a solver:

```
Model -> ParameterValues -> Geometry -> Mesh -> Discretisation -> Solver -> Solution
```

`pybamm.Simulation` runs that pipeline for you, and it works for *any* model,
not just batteries. Here is the simplest possible one, exponential decay
$\frac{dx}{dt} = -kx$, written from scratch. Day 2 builds a full battery model
in exactly this way.

In [10]:
x = pybamm.Variable("x")
k = pybamm.Parameter("Decay rate [s-1]")

decay = pybamm.BaseModel(name="exponential decay")
decay.rhs = {x: -k * x}
decay.initial_conditions = {x: 1}
decay.variables = {"x": x}

parameter_values = pybamm.ParameterValues({"Decay rate [s-1]": 0.1})
sol = pybamm.Simulation(decay, parameter_values=parameter_values).solve([0, 30])
plt.plot(sol.t, sol["x"](sol.t), label="PyBaMM")
plt.plot(sol.t, np.exp(-0.1 * sol.t), "--", label="exact")
plt.legend()

## Exercises

1. Compare the SPM and DFN at 3C using the `"Chen2020"` parameter set. Where do
   they disagree, and why?
2. Use `pybamm.lithium_ion.DFN({"thermal": "lumped"})` and plot
   `"Volume-averaged cell temperature [C]"` for a 1C and a 2C discharge.
3. Add a second equation to the decay model, e.g. $\frac{dy}{dt} = x$, and
   check the result against the exact solution.

## References

In [11]:
pybamm.print_citations()

[1] Joel A. E. Andersson, Joris Gillis, Greg Horn, James B. Rawlings, and Moritz Diehl. CasADi – A software framework for nonlinear optimization and optimal control. Mathematical Programming Computation, 11(1):1–36, 2019. doi:10.1007/s12532-018-0139-4.
[2] Von DAG Bruggeman. Berechnung verschiedener physikalischer konstanten von heterogenen substanzen. i. dielektrizitätskonstanten und leitfähigkeiten der mischkörper aus isotropen substanzen. Annalen der physik, 416(7):636–664, 1935.
[3] Chang-Hui Chen, Ferran Brosa Planella, Kieran O'Regan, Dominika Gastol, W. Dhammika Widanage, and Emma Kendrick. Development of Experimental Techniques for Parameterization of Multi-scale Lithium-ion Battery Models. Journal of The Electrochemical Society, 167(8):080534, 2020. doi:10.1149/1945-7111/ab9050.
[4] Marc Doyle, Thomas F. Fuller, and John Newman. Modeling of galvanostatic charge and discharge of the lithium/polymer/insertion cell. Journal of the Electrochemical society, 140(6):1526–1533, 1993. 